# Python basics: read and clean a CSV
A warm-up from [The Joinery](https://thejoinery.dev/). Run each cell with **Shift + Enter**. The full explanations are on the guide page: https://thejoinery.dev/#/projects/python-basics

In [ ]:
import pandas as pd

## 1. Read the file

In [ ]:
df = pd.read_csv(URL)
print(df.shape)
print(df.head())

## 2. Look before you clean

In [ ]:
df.info()

In [ ]:
print(df["Province"].value_counts())

## 3. Tidy the column names

In [ ]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
print(df.columns.tolist())

## 4. Fix the data types

In [ ]:
df["order_total"] = (df["order_total"]
                     .str.replace("R", "", regex=False)
                     .str.replace(" ", "", regex=False)
                     .astype(float))
for col in ["order_date", "delivered_date"]:
    iso = pd.to_datetime(df[col], format="%Y-%m-%d", errors="coerce")   # 2025-04-24
    dmy = pd.to_datetime(df[col], format="%d/%m/%Y", errors="coerce")   # 24/04/2025
    df[col] = iso.fillna(dmy)
print(df.dtypes)

## 5. Tidy messy text

In [ ]:
df["status"] = df["status"].str.strip().str.lower()
province_names = {
    "GP": "Gauteng", "WC": "Western Cape", "EC": "Eastern Cape",
    "KZN": "KwaZulu-Natal", "Kwazulu Natal": "KwaZulu-Natal",
}
df["province"] = df["province"].str.strip().replace(province_names)
df["province"] = df["province"].where(df["province"] == "KwaZulu-Natal", df["province"].str.title())
print(df["status"].value_counts())
print(df["province"].value_counts())

## 6. Deal with missing values

In [ ]:
print(df.isna().sum())

In [ ]:
df["city"] = df["city"].fillna("Unknown")
print(df[df["order_total"].isna()][["order_id", "status", "order_total"]].head(3))

## 7. Remove duplicate rows

In [ ]:
print(df.duplicated().sum())
df = df.drop_duplicates()
print(df.shape)

## 8. Catch impossible values, then save

In [ ]:
tests = df["order_id"].str.startswith("TEST")
negative = df["order_total"] < 0
time_travel = df["delivered_date"] < df["order_date"]
print(tests.sum(), negative.sum(), time_travel.sum())

In [ ]:
df = df[~tests & ~negative & ~time_travel]
print(df.shape)
df.to_csv("orders_clean.csv", index=False)